# Looksee — Deep Learning Image Analysis

This Colab notebook runs Looksee's vision-language image analysis using a pretrained deep-learning model through OpenRouter. It supports:

- General image analysis
- Identify Parts mode for visible mechanical, engine, electronic, appliance, and tool components
- JPG, PNG, and WebP uploads up to 10 MB
- Structured JSON output with confidence and safety notes

Your OpenRouter key is entered privately at runtime and is never saved in this notebook.


In [ ]:
!pip -q install pillow requests


In [ ]:
import base64
import io
import json
from getpass import getpass
from pathlib import Path

import requests
from PIL import Image
from IPython.display import display, JSON

MODEL = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MAX_FILE_SIZE = 10 * 1024 * 1024
ALLOWED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

print(f"Configured model: {MODEL}")
print("The API key will be requested privately in the next cell.")


In [ ]:
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key (input stays hidden): ").strip()
if not OPENROUTER_API_KEY:
    raise ValueError("An OpenRouter API key is required. Create one at https://openrouter.ai/keys.")
print("API key loaded securely for this Colab runtime only.")


In [ ]:
def build_prompt(analysis_mode: str) -> str:
    if analysis_mode not in {"general", "components"}:
        raise ValueError("analysis_mode must be 'general' or 'components'.")

    if analysis_mode == "components":
        component_instruction = (
            "This is COMPONENT IDENTIFICATION MODE. Carefully identify every clearly visible "
            "mechanical, engine, vehicle, appliance, electronic, tool, or other physical component. "
            "Use a precise common name only where visible. For engine photos, distinguish components "
            "such as valve cover, intake manifold, air filter housing, battery, radiator hose, oil cap, "
            "alternator, belt, coolant reservoir, fuse box, and wiring only when you can actually see them. "
            "Do not invent labels for hidden parts."
        )
    else:
        component_instruction = (
            "Also identify clearly visible physical components when useful, but keep the answer focused "
            "on the overall image."
        )

    schema = {
        "summary": "one concise factual sentence",
        "objects": ["up to eight visible objects or subjects"],
        "scene": "short scene or context description",
        "parts": [{
            "name": "clear component name",
            "category": "engine | vehicle | mechanical | electronic | appliance | tool | other",
            "location": "where it appears in the image",
            "confidence": "high | medium | low",
            "description": "one brief description of its visible role or appearance"
        }],
        "confidence_note": "brief uncertainty statement",
        "safety_note": "visual identification is informational only; verify before repair, safety decisions, or purchase"
    }
    return (
        "Analyze this image for Looksee, a general image and parts-identification application.\n"
        + component_instruction + "\n"
        + "Return ONLY valid JSON with exactly this structure:\n"
        + json.dumps(schema, indent=2) + "\n"
        + "If no physical components can be confidently identified, return an empty parts array. "
        + "Never claim model numbers, exact compatibility, internal faults, safety conditions, or repair "
        + "requirements from the image alone. Do not identify people or infer sensitive traits."
    )


In [ ]:
def parse_model_json(raw_text: str) -> dict:
    # Parse normal JSON or JSON wrapped in a Markdown code fence.
    if isinstance(raw_text, list):
        raw_text = "".join(
            item.get("text", "") for item in raw_text
            if isinstance(item, dict) and isinstance(item.get("text"), str)
        )
    clean = str(raw_text).strip()
    if clean.startswith("```"):
        lines = clean.splitlines()
        if lines and lines[0].strip().startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        clean = "\n".join(lines).strip()

    try:
        data = json.loads(clean)
    except json.JSONDecodeError:
        start, end = clean.find("{"), clean.rfind("}")
        if start < 0 or end <= start:
            raise ValueError("The model did not return valid JSON.")
        try:
            data = json.loads(clean[start:end + 1])
        except json.JSONDecodeError as exc:
            raise ValueError("The model returned unreadable JSON.") from exc

    if not isinstance(data, dict):
        raise ValueError("The model returned JSON, but not an analysis object.")
    data.setdefault("objects", [])
    data.setdefault("parts", [])
    data.setdefault("confidence_note", "Confidence is limited to what is visible in the image.")
    data.setdefault("safety_note", "Visual identification is informational only. Verify before repair or purchase.")
    return data


In [ ]:
def analyze_image(image_path: str, analysis_mode: str = "general") -> dict:
    # Upload one image to the deep-learning vision model and return structured analysis.
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")
    if path.suffix.lower() not in ALLOWED_EXTENSIONS:
        raise ValueError("Use a JPG, PNG, or WebP image.")
    if path.stat().st_size > MAX_FILE_SIZE:
        raise ValueError("Image must be 10 MB or smaller.")

    try:
        with Image.open(path) as source:
            source.verify()
        with Image.open(path) as source:
            normalized = source.convert("RGB")
            output = io.BytesIO()
            normalized.save(output, format="JPEG", quality=90, optimize=True)
            encoded = base64.b64encode(output.getvalue()).decode("utf-8")
    except (OSError, ValueError) as exc:
        raise ValueError("This file could not be read as an image. Try another JPG, PNG, or WebP.") from exc

    payload = {
        "model": MODEL,
        "temperature": 0.1,
        "max_tokens": 1200,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": build_prompt(analysis_mode)},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{encoded}"}},
            ],
        }],
    }
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://looksee.app",
        "X-Title": "Looksee Image Analysis",
    }

    try:
        response = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=90)
    except requests.RequestException as exc:
        raise RuntimeError("Could not reach OpenRouter. Check the Colab runtime connection.") from exc

    if response.status_code in {401, 403}:
        raise RuntimeError("OpenRouter rejected the API key. Check that the key is active and has credits or model access.")
    if response.status_code == 404:
        raise RuntimeError(f"The configured model is unavailable: {MODEL}")
    if response.status_code == 429:
        raise RuntimeError("OpenRouter rate-limited this request. Wait a minute and try again.")
    if response.status_code >= 400:
        try:
            detail = response.json().get("error", {}).get("message", response.text[:300])
        except ValueError:
            detail = response.text[:300]
        raise RuntimeError(f"OpenRouter request failed (HTTP {response.status_code}): {detail}")

    try:
        response_data = response.json()
        if "error" in response_data:
            raise RuntimeError(response_data["error"].get("message", "Unknown provider error"))
        content = response_data["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError, ValueError) as exc:
        raise RuntimeError("OpenRouter returned an unexpected response.") from exc

    return parse_model_json(content)


## Upload and analyze an image

Run the next cell, choose an image, and enter either `general` or `components`. The `components` option is Looksee's Identify Parts mode.


In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise ValueError("No image was uploaded.")

image_path = next(iter(uploaded))
analysis_mode = input("Analysis mode — type general or components: ").strip().lower()
if analysis_mode not in {"general", "components"}:
    raise ValueError("Choose exactly 'general' or 'components'.")

display(Image.open(image_path))
result = analyze_image(image_path, analysis_mode)
display(JSON(result))


## Optional: analyze another image

Run this cell again whenever you want to test another image. The key remains available only in the current Colab runtime.


In [ ]:
# Example after uploading another image manually:
# result = analyze_image("your_uploaded_filename.jpg", "components")
# display(JSON(result))
